# Driver-Independent Suppression of Nuclear-Encoded Mitochondrial Fatty Acid Oxidation in Breast Cancer: A Multi-Cohort Transcriptomic Analysis

**Authors:** Aroob A. Gomosani, Haneen A. Marghalani, Layal M. Al Matar  
**Institution:** King Abdulaziz University, Jeddah, Saudi Arabia  
**Correspondence:** agomosani0004@stu.kau.edu.sa

---

## Overview

Complete analysis pipeline for the BRCA mitochondrial transcriptomics study.

**Pipeline:**
1. TCGA-BRCA discovery — MitoCarta 3.0-restricted differential expression (Welch t-test + BH FDR)
2. PCA quality control — before any DE analysis
3. Stage I analysis — earliest-stage dysregulation (n=163 independent)
4. PAM50 stratification — four-subtype pan-subtype intersection
5. 55-gene core signature — three-way intersection, zero mixed-direction genes
6. Candidate scoring and locking — pre-specified before any clinical testing
7. Clinical, survival (KM + Cox), and ROC analyses
8. GSEA — MitoCarta pathway enrichment
9. TP53/PIK3CA mutation independence
10. Composite FAO suppression score
11. DESeq2 methodological cross-validation
12. External validation — GSE109169 and GSE42568
13. METABRIC prognostic validation (cBioPortal)
14. Figure and table generation

**Data sources (all publicly available):**
- TCGA-BRCA: [UCSC Xena](https://xenabrowser.net) — FPKM-UQ, STAR counts, somatic mutations, clinical
- MitoCarta 3.0: [Broad Institute](https://www.broadinstitute.org/mitocarta)
- GSE109169: [NCBI GEO](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE109169) (Affymetrix HuEx-1.0-ST, 25 pairs)
- GSE42568: [NCBI GEO](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE42568) (Affymetrix HG-U133 Plus 2.0, 104T+17N)
- METABRIC: [cBioPortal](https://www.cbioportal.org/study/summary?id=brca_metabric) (n=1,980)

**Environment:** Python 3.12 · Google Colab (recommended) or local Jupyter


## 1. Environment Setup

In [ ]:
# Install dependencies (run once per Colab session)
!pip install pydeseq2 gseapy lifelines GEOparse matplotlib_venn --quiet


In [ ]:
import os, gzip, re, warnings, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from io import StringIO
from scipy.stats import ttest_ind, mannwhitneyu, chi2_contingency, spearmanr, f_oneway
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'figure.dpi'       : 150,
    'savefig.dpi'      : 300,
    'savefig.bbox'     : 'tight',
    'savefig.facecolor': 'white',
})
print("Environment ready.")


## 2. Configuration

> **⚙️ Edit this cell before running.** Set `DATA_FOLDER` to the directory containing your data files.
>
> **Expected files in DATA_FOLDER:**
> - `Gene_Expression_Data_Complete.gz` (138.5 MB — log2 FPKM-UQ)
> - `Clinical_Data_Complete.gz`
> - `BRCA_phenotype_clinical` (PAM50, ER, HER2, survival)
> - `Human.MitoCarta3.0.xls`
> - `TCGA-BRCA.star_counts.tsv` (754.8 MB — raw counts for DESeq2)
> - `TCGA-BRCA.somaticmutation_wxs.tsv`
> - `GSE109169_series_matrix.txt.gz`
> - `GSE42568_series_matrix.txt.gz`


In [ ]:
# ── USER CONFIGURATION ─────────────────────────────────────────────────────────
# Google Colab: DATA_FOLDER = '/content/drive/MyDrive/Colab Notebooks/Oncology_Project/'
# Local:        DATA_FOLDER = '/home/user/brca_data/'

DATA_FOLDER = '/content/drive/MyDrive/Colab Notebooks/Oncology_Project/'  # <-- CHANGE THIS

# Mount Google Drive (Colab only — remove if running locally)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── File paths ─────────────────────────────────────────────────────────────────
EXPR_FILE      = DATA_FOLDER + 'Gene_Expression_Data_Complete.gz'
CLINICAL_FILE  = DATA_FOLDER + 'Clinical_Data_Complete.gz'
PHENOTYPE_FILE = DATA_FOLDER + 'BRCA_phenotype_clinical'
MITOCARTA_FILE = DATA_FOLDER + 'Human.MitoCarta3.0.xls'
COUNTS_FILE    = DATA_FOLDER + 'TCGA-BRCA.star_counts.tsv'
MUTATION_FILE  = DATA_FOLDER + 'TCGA-BRCA.somaticmutation_wxs.tsv'
GEO109_FILE    = DATA_FOLDER + 'GSE109169_series_matrix.txt.gz'
GEO42_FILE     = DATA_FOLDER + 'GSE42568_series_matrix.txt.gz'
OUT            = DATA_FOLDER  # output files saved here

# ── Analysis thresholds — LOCKED, do not change ────────────────────────────────
PADJ_THRESHOLD   = 0.05
LOG2FC_THRESHOLD = 1.0
FDR_ALPHA        = 0.05
RANDOM_SEED      = 42

# ── Clinical column names ──────────────────────────────────────────────────────
STAGE_COL    = 'ajcc_pathologic_stage.diagnoses'
T_STAGE_COL  = 'ajcc_pathologic_t.diagnoses'
N_STAGE_COL  = 'ajcc_pathologic_n.diagnoses'
SUBTYPE_COL  = 'PAM50Call_RNAseq'
ER_COL       = 'ER_Status_nature2012'
HER2_COL     = 'HER2_Final_Status_nature2012'
AGE_COL      = 'age_at_diagnosis.diagnoses'
OS_TIME_COL  = 'OS_Time_nature2012'
OS_EVENT_COL = 'OS_event_nature2012'

SUBTYPES       = ['LumA', 'LumB', 'Basal', 'Her2']
STAGE_I_LABELS = ['Stage I', 'Stage IA', 'Stage IB']

# ── File existence check ───────────────────────────────────────────────────────
files = [
    ('Expression',  EXPR_FILE),
    ('Clinical',    CLINICAL_FILE),
    ('Phenotype',   PHENOTYPE_FILE),
    ('MitoCarta',   MITOCARTA_FILE),
    ('STAR counts', COUNTS_FILE),
    ('Mutations',   MUTATION_FILE),
    ('GSE109169',   GEO109_FILE),
    ('GSE42568',    GEO42_FILE),
]
all_ok = True
for label, path in files:
    if os.path.exists(path):
        print(f"  ✓  {label:14s}  {os.path.basename(path)}")
    else:
        all_ok = False
        print(f"  ✗  {label:14s}  MISSING — {path}")
print("\nAll files found. Ready to run." if all_ok else "\n⚠  Fix missing files before continuing.")


## 3. Data Loading

In [ ]:
# Load expression matrix (log2 FPKM-UQ + 1, already normalised)
print("Loading expression data...")
expression = pd.read_csv(EXPR_FILE, sep='\t', compression='gzip')
sample_cols = [c for c in expression.columns if c.startswith('TCGA')]
vals = expression[sample_cols].values
print(f"Shape: {expression.shape}")
print(f"Value range: {vals.min():.2f} – {vals.max():.2f}  (expect 0–22, mean ~3–4)")
print(f"Zero fraction: {(vals==0).mean()*100:.1f}%")


In [ ]:
# Load clinical data and PAM50 phenotype file; merge on 15-char barcode prefix
clinical = pd.read_csv(CLINICAL_FILE, sep='\t', compression='gzip')
clinical['sample_short'] = clinical['sample'].str[:15]

subtypes     = pd.read_csv(PHENOTYPE_FILE, sep='\t')
subtypes_slim = subtypes[[c for c in [
    'sampleID', 'PAM50Call_RNAseq', 'ER_Status_nature2012',
    'PR_Status_nature2012', 'HER2_Final_Status_nature2012',
    'OS_Time_nature2012', 'OS_event_nature2012',
] if c in subtypes.columns]].copy()
subtypes_slim['sample_short'] = subtypes_slim['sampleID'].str[:15]

clinical_complete = clinical.merge(
    subtypes_slim.drop(columns=['sampleID']), on='sample_short', how='left')
clinical_complete.to_csv(OUT + 'BRCA_clinical_complete.csv', index=False)

# Duplicate-safe lookup — prevents Series ambiguity errors downstream
clinical_lookup = clinical_complete.drop_duplicates('sample_short').set_index('sample_short')

print(f"Clinical: {clinical.shape}  |  Phenotype: {subtypes.shape}  |  Merged: {clinical_complete.shape}")
print(f"PAM50 available: {clinical_complete[SUBTYPE_COL].notna().sum()}")
print(f"\nPAM50 distribution:\n{clinical_lookup[SUBTYPE_COL].value_counts()}")


## 4. MitoCarta 3.0 Filtering and Sample Classification

Barcode position 13–14 encodes sample type: `01` = primary tumour, `11` = normal-adjacent, `06` = metastatic (excluded).


In [ ]:
mitocarta = pd.read_excel(MITOCARTA_FILE, sheet_name='A Human MitoCarta3.0')
mitocarta_set   = set(mitocarta['EnsemblGeneID_mapping_version_20200130'].dropna())
mitocarta_names = (mitocarta[['EnsemblGeneID_mapping_version_20200130','Symbol']]
                   .dropna()
                   .rename(columns={'EnsemblGeneID_mapping_version_20200130':'ensembl_id'}))
print(f"MitoCarta 3.0 genes: {len(mitocarta_set)}")

# Strip version suffix (GDC: ENSG00000000003.15 → ENSG00000000003)
expression['Ensembl_clean'] = expression['Ensembl_ID'].str.split('.').str[0]
expression_mito = expression[expression['Ensembl_clean'].isin(mitocarta_set)].copy()
print(f"Matched: {len(expression_mito)} / {len(mitocarta_set)} ({len(expression_mito)/len(mitocarta_set)*100:.1f}%)")

# Classify samples
all_samples   = [c for c in expression_mito.columns if c.startswith('TCGA')]
gene_cols     = ['Ensembl_ID', 'Ensembl_clean']
tumor_cols    = [c for c in all_samples if c[13:15] == '01']
normal_cols   = [c for c in all_samples if c[13:15] == '11']
met_cols      = [c for c in all_samples if c[13:15] == '06']
expression_tumor  = expression_mito[gene_cols + tumor_cols].copy()
expression_normal = expression_mito[gene_cols + normal_cols].copy()

print(f"\nPrimary tumour:  {len(tumor_cols)}")   # expected: 1106
print(f"Normal-adjacent: {len(normal_cols)}")   # expected: 113
print(f"Metastatic:      {len(met_cols)} (excluded)")
aligned = sum(1 for c in tumor_cols if c[:15] in clinical_lookup.index)
print(f"Clinical alignment: {aligned}/{len(tumor_cols)} ({100*aligned/len(tumor_cols):.1f}%)")


## 5. Quality Control — PCA

PCA is run **before** any differential expression analysis to confirm tumour-normal
separation and absence of technical batch effects. PC1 should separate the two groups.


In [ ]:
print("Running PCA on 1,219 samples × 1,079 genes...")
t_matrix = expression_tumor[tumor_cols].T.values.astype(float)
n_matrix = expression_normal[normal_cols].T.values.astype(float)
combined = np.vstack([t_matrix, n_matrix])
labels   = ['Tumour'] * len(tumor_cols) + ['Normal'] * len(normal_cols)

scaled = StandardScaler().fit_transform(combined)
pca    = PCA(n_components=2, random_state=RANDOM_SEED)
coords = pca.fit_transform(scaled)
var1, var2 = pca.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(8, 6))
for label, colour in [('Tumour','#c0392b'), ('Normal','#1a7a4a')]:
    mask = [l == label for l in labels]
    ax.scatter(coords[mask,0], coords[mask,1],
               c=colour, alpha=0.5, s=15, label=label, linewidths=0)
ax.set_xlabel(f'PC1 ({var1:.1f}% variance explained)')
ax.set_ylabel(f'PC2 ({var2:.1f}% variance explained)')
ax.set_title('PCA — Mitochondrial Gene Expression\nBRCA Tumours vs Normal-Adjacent', fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(OUT + 'BRCA_PCA.png', dpi=300)
plt.show()
print(f"PC1 = {var1:.1f}%  |  PC2 = {var2:.1f}%  (expected: PC1 ≥ 20%, clear separation)")


## 6. All-Stage Differential Expression

Welch's t-test (unequal variance) + Benjamini-Hochberg FDR across 1,079 MitoCarta genes.  
Dual threshold: **adjusted p < 0.05 AND |log₂FC| > 1.0**.


In [ ]:
print(f"Running DE: {len(tumor_cols)} tumours vs {len(normal_cols)} normals...")
results = []
for _, row in expression_mito.iterrows():
    eid = row['Ensembl_clean']
    t_v = expression_tumor.loc[expression_tumor['Ensembl_clean']==eid, tumor_cols].values.flatten().astype(float)
    n_v = expression_normal.loc[expression_normal['Ensembl_clean']==eid, normal_cols].values.flatten().astype(float)
    if len(t_v)==0 or len(n_v)==0: continue
    _, p = ttest_ind(t_v, n_v, equal_var=False)
    if np.isnan(p): p = 1.0
    results.append({'ensembl_id': eid,
                    'log2fc'    : np.mean(t_v) - np.mean(n_v),
                    'pvalue'    : p})

de_all = pd.DataFrame(results)
_, adj_p,_,_ = multipletests(de_all['pvalue'], alpha=FDR_ALPHA, method='fdr_bh')
de_all['adjusted_pvalue'] = adj_p
de_all = de_all.merge(mitocarta_names, on='ensembl_id', how='left')

de_sig = de_all[
    (de_all['adjusted_pvalue'] < PADJ_THRESHOLD) &
    (de_all['log2fc'].abs() > LOG2FC_THRESHOLD)
].copy()

print(f"\nDE RESULTS:")
print(f"  Tested:        {len(de_all)}")
print(f"  Significant:   {len(de_sig)}")  # expected: 126
print(f"  Upregulated:   {(de_sig['log2fc']>0).sum()}")
print(f"  Downregulated: {(de_sig['log2fc']<0).sum()}")
print(f"\nTop 5 by significance:")
print(de_sig.nsmallest(5,'adjusted_pvalue')[['Symbol','log2fc','adjusted_pvalue']].to_string())

de_all.to_csv(OUT + 'BRCA_DE_results_all_genes_annotated.csv', index=False)
de_sig.to_csv(OUT + 'BRCA_DE_results_significant_annotated.csv', index=False)
print("\nSaved.")


In [ ]:
# ── Candidate gene dictionary (pre-specified) ──────────────────────────────────
CANDIDATES = {
    'GLYAT'  : 'ENSG00000149124', 'ALDH1L1': 'ENSG00000144908',
    'ACADL'  : 'ENSG00000115361', 'ABCA9'  : 'ENSG00000154258',
    'GPAM'   : 'ENSG00000119927', 'PDK4'   : 'ENSG00000004799',
    'ABCD2'  : 'ENSG00000173208', 'MAOA'   : 'ENSG00000189221',
    'PDE2A'  : 'ENSG00000186642', 'ACACB'  : 'ENSG00000076555',
}

# ── All-stage volcano ─────────────────────────────────────────────────────────
min_p = de_all['adjusted_pvalue'][de_all['adjusted_pvalue']>0].min()
de_all['neg_log10_p'] = -np.log10(de_all['adjusted_pvalue'].replace(0, min_p))
de_all['rank_score']  = de_all['neg_log10_p'] * np.sign(de_all['log2fc'])

up   = (de_all['adjusted_pvalue'] < PADJ_THRESHOLD) & (de_all['log2fc'] >  LOG2FC_THRESHOLD)
down = (de_all['adjusted_pvalue'] < PADJ_THRESHOLD) & (de_all['log2fc'] < -LOG2FC_THRESHOLD)
ns   = ~(up|down)

fig, ax = plt.subplots(figsize=(9,7))
ax.scatter(de_all.loc[ns,'log2fc'],   de_all.loc[ns,'neg_log10_p'],   c='#aaaaaa', alpha=0.4, s=12)
ax.scatter(de_all.loc[up,'log2fc'],   de_all.loc[up,'neg_log10_p'],   c='#c0392b', alpha=0.7, s=15)
ax.scatter(de_all.loc[down,'log2fc'], de_all.loc[down,'neg_log10_p'], c='#1a5276', alpha=0.7, s=15)
for gene, eid in CANDIDATES.items():
    row = de_all[de_all['ensembl_id']==eid]
    if len(row):
        ax.scatter(row['log2fc'], row['neg_log10_p'],
                   c='#f39c12', s=55, zorder=5, edgecolors='black', linewidths=0.5)
        ax.annotate(gene, (row['log2fc'].values[0], row['neg_log10_p'].values[0]),
                    xytext=(5,3), textcoords='offset points', fontsize=7, fontweight='bold')
ax.axhline(-np.log10(PADJ_THRESHOLD), color='k', linestyle='--', lw=0.8, alpha=0.5)
ax.axvline( LOG2FC_THRESHOLD, color='k', linestyle=':', lw=0.8, alpha=0.5)
ax.axvline(-LOG2FC_THRESHOLD, color='k', linestyle=':', lw=0.8, alpha=0.5)
ax.set_xlabel('log₂ Fold Change (Tumour vs Normal)')
ax.set_ylabel('−log₁₀ (Adjusted p-value)')
ax.set_title(f'BRCA Mitochondrial DE — All Stage\n{up.sum()} up  {down.sum()} down', fontweight='bold')
plt.tight_layout()
plt.savefig(OUT + 'BRCA_volcano_allstage.png', dpi=300)
plt.show()


## 7. Stage I Analysis

Tests whether mitochondrial dysregulation is present at the earliest diagnosable stage.
The 21 patients contributing both tumour and normal samples are excluded (independence check).


In [ ]:
# Build stage lookup using duplicate-safe clinical_lookup
stage_lookup = {}
for barcode in tumor_cols:
    short = barcode[:15]
    if short in clinical_lookup.index:
        val = clinical_lookup.loc[short, STAGE_COL]
        if pd.notna(val): stage_lookup[barcode] = val

stage_series = pd.Series(stage_lookup)
stage1_all   = stage_series[stage_series.isin(STAGE_I_LABELS)].index.tolist()
print(f"Stage distribution:\n{stage_series.value_counts()}")
print(f"\nStage I total: {len(stage1_all)}")

# Independence check
normal_patient_ids = set(c[:12] for c in normal_cols)
stage1_indep = [c for c in stage1_all if c[:12] not in normal_patient_ids]
print(f"Stage I independent (no matched normal): {len(stage1_indep)}")  # expected: 163

# Run Stage I DE
results_s1 = []
for _, row in expression_mito.iterrows():
    eid  = row['Ensembl_clean']
    s1_v = expression_tumor.loc[expression_tumor['Ensembl_clean']==eid, stage1_indep].values.flatten().astype(float)
    n_v  = expression_normal.loc[expression_normal['Ensembl_clean']==eid, normal_cols].values.flatten().astype(float)
    if len(s1_v) < 5: continue
    _, p = ttest_ind(s1_v, n_v, equal_var=False)
    if np.isnan(p): p = 1.0
    results_s1.append({'ensembl_id': eid,
                       'log2fc_s1' : np.mean(s1_v) - np.mean(n_v),
                       'pvalue_s1' : p})

s1_df = pd.DataFrame(results_s1)
_, adj1,_,_ = multipletests(s1_df['pvalue_s1'], alpha=FDR_ALPHA, method='fdr_bh')
s1_df['adjusted_pvalue_s1'] = adj1
s1_df = s1_df.merge(mitocarta_names, on='ensembl_id', how='left')
s1_sig = s1_df[(s1_df['adjusted_pvalue_s1'] < PADJ_THRESHOLD) &
               (s1_df['log2fc_s1'].abs() > LOG2FC_THRESHOLD)]

overlap = set(de_sig['ensembl_id']) & set(s1_sig['ensembl_id'])
print(f"\nStage I significant: {len(s1_sig)}  (expected: 109)")
print(f"Overlap with all-stage: {len(overlap)}/{len(s1_sig)} ({len(overlap)/max(len(s1_sig),1)*100:.1f}%)")

# Sensitivity: rerun WITH overlapping patients — should be >95% concordant
res_sens = []
for _, row in expression_mito.iterrows():
    eid = row['Ensembl_clean']
    sv  = expression_tumor.loc[expression_tumor['Ensembl_clean']==eid, stage1_all].values.flatten().astype(float)
    nv  = expression_normal.loc[expression_normal['Ensembl_clean']==eid, normal_cols].values.flatten().astype(float)
    if len(sv)<5: continue
    _, p = ttest_ind(sv, nv, equal_var=False)
    if np.isnan(p): p=1.0
    res_sens.append({'ensembl_id':eid,'p_sens':p,'fc_sens':np.mean(sv)-np.mean(nv)})
sens_df = pd.DataFrame(res_sens)
_, adjps,_,_ = multipletests(sens_df['p_sens'], alpha=FDR_ALPHA, method='fdr_bh')
sens_df['adj_p_sens'] = adjps
sens_sig = sens_df[(sens_df['adj_p_sens']<PADJ_THRESHOLD)&(sens_df['fc_sens'].abs()>LOG2FC_THRESHOLD)]
sens_overlap = set(s1_sig['ensembl_id']) & set(sens_sig['ensembl_id'])
print(f"Sensitivity (full Stage I): {len(sens_overlap)/max(len(s1_sig),1)*100:.1f}% concordance")

s1_df.to_csv(OUT + 'BRCA_Stage1_DE_all_genes.csv', index=False)
s1_sig.to_csv(OUT + 'BRCA_Stage1_DE_significant.csv', index=False)
print("Saved Stage I results.")


## 8. PAM50 Subtype Stratification

DE run independently within each PAM50 subtype vs the same 113 normals.
Zero mixed-direction genes in the pan-subtype intersection = signal is universal,
not driven by a single subtype.  Normal-like subtype (n=23) excluded.


In [ ]:
subtype_map = {}
for barcode in tumor_cols:
    short = barcode[:15]
    if short in clinical_lookup.index:
        val = clinical_lookup.loc[short, SUBTYPE_COL]
        if pd.notna(val): subtype_map[barcode] = val

subtype_series = pd.Series(subtype_map)
print(f"Subtype distribution:\n{subtype_series.value_counts()}")

subtype_results = {}
for subtype in SUBTYPES:
    sub_samples = subtype_series[subtype_series==subtype].index.tolist()
    if len(sub_samples) < 10:
        print(f"  Skip {subtype}: n={len(sub_samples)}"); continue
    print(f"  {subtype}: n={len(sub_samples)}", end=' ')
    sub_res = []
    for _, row in expression_mito.iterrows():
        eid = row['Ensembl_clean']
        sv  = expression_tumor.loc[expression_tumor['Ensembl_clean']==eid, sub_samples].values.flatten().astype(float)
        nv  = expression_normal.loc[expression_normal['Ensembl_clean']==eid, normal_cols].values.flatten().astype(float)
        _, p = ttest_ind(sv, nv, equal_var=False)
        if np.isnan(p): p=1.0
        sub_res.append({'ensembl_id':eid,'log2fc':np.mean(sv)-np.mean(nv),'pvalue':p})
    sub_df = pd.DataFrame(sub_res)
    _, adjp,_,_ = multipletests(sub_df['pvalue'], alpha=FDR_ALPHA, method='fdr_bh')
    sub_df['adjusted_pvalue'] = adjp
    sub_sig = sub_df[(sub_df['adjusted_pvalue']<PADJ_THRESHOLD)&(sub_df['log2fc'].abs()>LOG2FC_THRESHOLD)]
    subtype_results[subtype] = sub_sig
    print(f"→ {len(sub_sig)} significant")
    sub_sig.to_csv(OUT + f'BRCA_DE_{subtype}_significant.csv', index=False)

# Pan-subtype intersection
pan_ids = set.intersection(*[set(df['ensembl_id']) for df in subtype_results.values()])
print(f"\nPan-subtype intersection: {len(pan_ids)} genes")  # expected: 57

# Direction consistency — the critical result
direction_map = {}
for eid in pan_ids:
    signs = [np.sign(df[df['ensembl_id']==eid].iloc[0]['log2fc'])
             for df in subtype_results.values() if len(df[df['ensembl_id']==eid])]
    direction_map[eid] = ('Consistent DOWN' if all(s<0 for s in signs) else
                          'Consistent UP'   if all(s>0 for s in signs) else 'MIXED')

print(f"  DOWN: {sum(1 for v in direction_map.values() if v=='Consistent DOWN')}")
print(f"  UP:   {sum(1 for v in direction_map.values() if v=='Consistent UP')}")
print(f"  MIXED: {sum(1 for v in direction_map.values() if v=='MIXED')}  ← must be 0")


## 9. Core Signature and Candidate Scoring

**Three-way intersection:** all-stage ∩ Stage I ∩ pan-PAM50 → **55-gene core signature**.

Pre-specified scoring formula (applied before any clinical testing):
```
score = 0.40 × |FC all-stage| + 0.40 × min(|FC per subtype|) + 0.20 × (1 / (1 + SD across subtypes))
```

**Top 10 genes are locked at the end of this cell.**


In [ ]:
all_stage_ids  = set(de_sig['ensembl_id'])
stage1_ids     = set(s1_sig['ensembl_id'])
pansubtype_ids = pan_ids

core_ids = all_stage_ids & stage1_ids & pansubtype_ids
core_df  = de_sig[de_sig['ensembl_id'].isin(core_ids)].copy()
core_df['direction'] = core_df['ensembl_id'].map(direction_map)

print(f"All-stage:    {len(all_stage_ids)}")
print(f"Stage I:      {len(stage1_ids)}")
print(f"Pan-PAM50:    {len(pansubtype_ids)}")
print(f"CORE (3-way): {len(core_ids)}")  # expected: 55
print(f"  DOWN: {(core_df['direction']=='Consistent DOWN').sum()}")
print(f"  UP:   {(core_df['direction']=='Consistent UP').sum()}")
print(f"  MIXED:{(core_df['direction']=='MIXED').sum()}  ← must be 0")

# Pre-specified scoring — LOCKED
scores = []
for _, row in core_df.iterrows():
    eid     = row['ensembl_id']
    all_fc  = abs(row['log2fc'])
    sub_fcs = [abs(df[df['ensembl_id']==eid].iloc[0]['log2fc'])
               for df in subtype_results.values() if len(df[df['ensembl_id']==eid])]
    if not sub_fcs: continue
    score = 0.40*all_fc + 0.40*min(sub_fcs) + 0.20*(1/(1+np.std(sub_fcs)))
    scores.append({'ensembl_id':eid,'score':round(score,3),
                   'all_stage_fc':round(all_fc,3),'min_subtype_fc':round(min(sub_fcs),3)})

score_df = pd.DataFrame(scores).merge(mitocarta_names, on='ensembl_id', how='left')
score_df  = score_df.sort_values('score', ascending=False).reset_index(drop=True)

print(f"\nTOP 10 CANDIDATES (locked):")
print(score_df.head(10)[['Symbol','score','all_stage_fc','min_subtype_fc']].to_string())

# ── LOCK ───────────────────────────────────────────────────────────────────────
LOCKED_CANDIDATES_EID  = score_df.head(10)['ensembl_id'].tolist()
LOCKED_CANDIDATES_SYM  = score_df.head(10)['Symbol'].tolist()
# Verify against pre-specified CANDIDATES dict
expected = set(CANDIDATES.keys())
top10    = set(score_df.head(10)['Symbol'].dropna())
print(f"\nMatch with pre-specified CANDIDATES: {len(expected & top10)}/10")  # expected: 9/10

core_df.to_csv(OUT + 'BRCA_core_55gene_signature.csv', index=False)
score_df.to_csv(OUT + 'BRCA_core_55gene_ranked.csv', index=False)
print("Saved core signature and ranked candidates.")


## 10. Expression Heatmap

Z-scored relative to **normal tissue mean and SD** (correct approach).  
Row z-score is intentionally avoided as it hides direction of effect.


In [ ]:
top50_eids = de_sig.nsmallest(50, 'adjusted_pvalue')['ensembl_id'].tolist()
top50_syms = {row['ensembl_id']: row['Symbol']
              for _, row in de_sig.iterrows()
              if row['ensembl_id'] in top50_eids and pd.notna(row['Symbol'])}

rows_hm = []
for eid in top50_eids:
    t_v = expression_tumor.loc[expression_tumor['Ensembl_clean']==eid, tumor_cols].values.flatten().astype(float)
    n_v = expression_normal.loc[expression_normal['Ensembl_clean']==eid, normal_cols].values.flatten().astype(float)
    z   = (t_v - np.mean(n_v)) / (np.std(n_v) + 1e-8)
    rows_hm.append(pd.Series(z, index=tumor_cols, name=top50_syms.get(eid, eid)))
heatmap_z = pd.DataFrame(rows_hm)

pam50_colors = {'LumA':'#2196F3','LumB':'#4CAF50','Basal':'#F44336','Her2':'#9C27B0'}
col_colors   = pd.Series(
    [pam50_colors.get(str(clinical_lookup.loc[c[:15], SUBTYPE_COL])
                      if c[:15] in clinical_lookup.index else '', '#eeeeee')
     for c in tumor_cols], index=tumor_cols)

g = sns.clustermap(
    heatmap_z, col_colors=col_colors, col_cluster=True, row_cluster=True,
    cmap='RdBu_r', vmin=-3, vmax=3, xticklabels=False, yticklabels=True,
    figsize=(18, 10), linewidths=0, cbar_pos=(0.02, 0.85, 0.03, 0.12))
g.ax_heatmap.set_xlabel(f'BRCA tumour samples (n={len(tumor_cols)}, ordered by PAM50)')
g.ax_heatmap.tick_params(axis='y', labelsize=8)
g.cax.set_ylabel('Z-score (vs normal mean/SD)', fontsize=9, rotation=90, labelpad=10)

legend_patches = [mpatches.Patch(color=c, label=l) for l,c in pam50_colors.items()]
legend_patches.append(mpatches.Patch(color='#eeeeee', label='No PAM50 call'))
g.ax_col_colors.legend(handles=legend_patches, title='PAM50',
                        loc='lower center', bbox_to_anchor=(0.5,-8), ncol=3, fontsize=8)
g.fig.suptitle('Top 50 DE Mitochondrial Genes — BRCA\n'
               'Z-scored relative to normal-adjacent tissue (blue=suppressed, red=elevated)',
               fontsize=12, fontweight='bold', y=1.01)
plt.savefig(OUT + 'BRCA_heatmap_top50.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Heatmap saved.")


## 11. Clinical Associations

In [ ]:
# 70 tests (10 genes × 7 variables), BH FDR across ALL simultaneously
clin_data = []
for barcode in tumor_cols:
    short = barcode[:15]
    if short not in clinical_lookup.index: continue
    row = clinical_lookup.loc[short]
    clin_data.append({
        'barcode'     : barcode,
        'PAM50'       : row.get(SUBTYPE_COL,  np.nan),
        'AJCC_stage'  : row.get(STAGE_COL,    np.nan),
        'T_stage'     : row.get(T_STAGE_COL,  np.nan),
        'nodal_status': row.get(N_STAGE_COL,  np.nan),
        'ER_status'   : row.get(ER_COL,       np.nan),
        'HER2_status' : row.get(HER2_COL,     np.nan),
        'age_days'    : row.get(AGE_COL,      np.nan),
    })
clin_df = pd.DataFrame(clin_data)
clin_df['age'] = pd.to_numeric(clin_df['age_days'], errors='coerce') / 365.25

cat_vars    = ['PAM50','AJCC_stage','T_stage','nodal_status','ER_status','HER2_status']
all_results = []
for gene, eid in CANDIDATES.items():
    expr_vals  = expression_tumor.loc[expression_tumor['Ensembl_clean']==eid, tumor_cols].values.flatten()
    expr_group = (pd.Series(expr_vals, index=tumor_cols) > np.median(expr_vals)).astype(int)
    clin_df['gene_group'] = clin_df['barcode'].map(expr_group.to_dict())
    for var in cat_vars:
        valid = clin_df[['gene_group',var]].dropna()
        if len(valid)<20 or valid[var].nunique()<2:
            all_results.append({'gene':gene,'variable':var,'test':'Chi-square','pvalue':np.nan,'n':len(valid)})
            continue
        _, p,_,_ = chi2_contingency(pd.crosstab(valid['gene_group'], valid[var]))
        all_results.append({'gene':gene,'variable':var,'test':'Chi-square','pvalue':p,'n':len(valid)})
    valid_age = clin_df[['gene_group','age']].dropna()
    h_a = valid_age[valid_age['gene_group']==1]['age'].values
    l_a = valid_age[valid_age['gene_group']==0]['age'].values
    p_age = mannwhitneyu(h_a, l_a, alternative='two-sided')[1] if (len(h_a)>5 and len(l_a)>5) else np.nan
    all_results.append({'gene':gene,'variable':'age','test':'Mann-Whitney U','pvalue':p_age,'n':len(valid_age)})

assoc_df = pd.DataFrame(all_results)
valid_mask = assoc_df['pvalue'].notna()
_, adj_p,_,_ = multipletests(assoc_df.loc[valid_mask,'pvalue'], alpha=FDR_ALPHA, method='fdr_bh')
assoc_df.loc[valid_mask,'adjusted_pvalue'] = adj_p
assoc_df['significant'] = assoc_df['adjusted_pvalue'] < FDR_ALPHA

print(f"Significant: {assoc_df['significant'].sum()} / {len(assoc_df)}")  # expected: 41
print(f"\nBreadth per gene:")
print(assoc_df.groupby('gene')['significant'].sum().sort_values(ascending=False))
assoc_df.to_csv(OUT + 'BRCA_association_results_complete.csv', index=False)


## 12. Survival Analysis — Kaplan-Meier and Cox Regression

In [ ]:
surv_data = []
for barcode in tumor_cols:
    short = barcode[:15]
    if short not in clinical_lookup.index: continue
    row = clinical_lookup.loc[short]
    surv_data.append({
        'barcode'  : barcode,
        'OS_time'  : pd.to_numeric(row.get(OS_TIME_COL,  np.nan), errors='coerce'),
        'OS_event' : pd.to_numeric(row.get(OS_EVENT_COL, np.nan), errors='coerce'),
        'age'      : pd.to_numeric(row.get(AGE_COL,      np.nan), errors='coerce') / 365.25,
        'stage'    : row.get(STAGE_COL,   np.nan),
        'PAM50'    : row.get(SUBTYPE_COL, np.nan),
        'ER_status': row.get(ER_COL,      np.nan),
    })
surv_df = pd.DataFrame(surv_data).dropna(subset=['OS_time','OS_event'])
print(f"Survival: {len(surv_df)} patients, {int(surv_df['OS_event'].sum())} events "
      f"({surv_df['OS_event'].mean()*100:.1f}%)")
# Note: ~14% events → underpowered for Cox. Null Cox = initiation marker, not absent biology.

# Kaplan-Meier — all 10 candidates
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes = axes.flatten()
km_results = []
for idx, (gene, eid) in enumerate(CANDIDATES.items()):
    expr_vals   = expression_tumor.loc[expression_tumor['Ensembl_clean']==eid, tumor_cols].values.flatten()
    expr_series = pd.Series(expr_vals, index=tumor_cols)
    sg = surv_df.copy(); sg['expr'] = sg['barcode'].map(expr_series.to_dict()); sg = sg.dropna(subset=['expr'])
    med = np.median(sg['expr'])
    high = sg[sg['expr']>med]; low = sg[sg['expr']<=med]
    kmf_h = KaplanMeierFitter(); kmf_l = KaplanMeierFitter()
    kmf_h.fit(high['OS_time'], high['OS_event'], label='High')
    kmf_l.fit(low['OS_time'],  low['OS_event'],  label='Low')
    res = logrank_test(high['OS_time'], low['OS_time'],
                       event_observed_A=high['OS_event'], event_observed_B=low['OS_event'])
    ax = axes[idx]
    kmf_h.plot_survival_function(ax=ax, ci_show=False, color='#2196F3', linewidth=1.5)
    kmf_l.plot_survival_function(ax=ax, ci_show=False, color='#e53935', linewidth=1.5)
    ax.set_title(f'{gene}\np = {res.p_value:.3f}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Time (days)', fontsize=8); ax.set_ylabel('Survival', fontsize=8)
    ax.legend(fontsize=7); ax.set_ylim(0,1)
    km_results.append({'gene':gene,'log_rank_p':res.p_value,'n_high':len(high),'n_low':len(low)})
plt.suptitle('KM Overall Survival — 10 Candidate Genes (TCGA-BRCA)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(OUT + 'BRCA_KM_all10_genes.png', dpi=300)
plt.show()

km_df = pd.DataFrame(km_results)
km_df.to_csv(OUT + 'BRCA_KM_results.csv', index=False)
print(km_df[['gene','log_rank_p']].sort_values('log_rank_p').to_string())


In [ ]:
# Cox regression — multivariate, adjusted for age/stage/PAM50/ER
cox_results = []
for gene, eid in CANDIDATES.items():
    expr_vals   = expression_tumor.loc[expression_tumor['Ensembl_clean']==eid, tumor_cols].values.flatten()
    expr_series = pd.Series(expr_vals, index=tumor_cols)
    cd = surv_df.copy()
    cd['expr'] = cd['barcode'].map(expr_series.to_dict())
    cd = cd.dropna(subset=['expr','age','stage','PAM50','ER_status'])
    cd['expr_z'] = (cd['expr'] - cd['expr'].mean()) / cd['expr'].std()
    cd = pd.get_dummies(cd, columns=['stage','PAM50','ER_status'], drop_first=True)
    feat = ['expr_z','age'] + [c for c in cd.columns if c.startswith(('stage_','PAM50_','ER_status_'))]
    inp  = cd[feat + ['OS_time','OS_event']].dropna()
    try:
        cph = CoxPHFitter()
        cph.fit(inp, duration_col='OS_time', event_col='OS_event')
        s = cph.summary.loc['expr_z']
        cox_results.append({'gene':gene,'HR':round(s['exp(coef)'],3),'p':s['p'],
                            'n':len(inp),'events':int(inp['OS_event'].sum())})
    except:
        cox_results.append({'gene':gene,'HR':np.nan,'p':np.nan,'n':0,'events':0})

cox_df = pd.DataFrame(cox_results)
_, adjc,_,_ = multipletests(cox_df['p'].fillna(1), alpha=FDR_ALPHA, method='fdr_bh')
cox_df['adjusted_p'] = adjc
cox_df['significant'] = cox_df['adjusted_p'] < FDR_ALPHA
print(f"Cox significant (BH FDR): {cox_df['significant'].sum()}/10")
print(cox_df[['gene','HR','p','adjusted_p','n','events']].to_string())
cox_df.to_csv(OUT + 'BRCA_cox_results.csv', index=False)


## 13. ROC Analysis

In [ ]:
y_true  = np.array([1]*len(tumor_cols) + [0]*len(normal_cols))
colours = plt.cm.tab10(np.linspace(0, 1, 10))
roc_results = []

fig, ax = plt.subplots(figsize=(9, 7))
for idx, (gene, eid) in enumerate(CANDIDATES.items()):
    t_expr  = expression_tumor.loc[expression_tumor['Ensembl_clean']==eid, tumor_cols].values.flatten()
    n_expr  = expression_normal.loc[expression_normal['Ensembl_clean']==eid, normal_cols].values.flatten()
    y_score = np.concatenate([t_expr, n_expr])
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    if roc_auc < 0.5:   # gene is suppressed — flip
        roc_auc = 1 - roc_auc; fpr, tpr = fpr[::-1], tpr[::-1]
    ax.plot(fpr, tpr, color=colours[idx], linewidth=1.5, label=f'{gene} ({roc_auc:.3f})')
    roc_results.append({'gene':gene,'AUC':round(roc_auc,3)})

ax.plot([0,1],[0,1],'k--', linewidth=0.8, alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — 10 Candidate Genes vs Normal', fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig(OUT + 'BRCA_ROC_curves.png', dpi=300)
plt.show()

roc_df = pd.DataFrame(roc_results).sort_values('AUC', ascending=False)
print(roc_df.to_string(index=False))
print(f"\nAUC > 0.90: {(roc_df['AUC']>0.90).sum()}/10")
roc_df.to_csv(OUT + 'BRCA_ROC_results.csv', index=False)


## 14. GSEA Pathway Enrichment

In [ ]:
import gseapy as gp

ranked_series = (de_all.dropna(subset=['Symbol'])
                 .sort_values('rank_score', ascending=False)
                 .set_index('Symbol')['rank_score'])
ranked_series = ranked_series[~ranked_series.index.duplicated(keep='first')]
print(f"Ranked list: {len(ranked_series)} genes")

# Load MitoCarta pathway gene sets (sheet C MitoPathways)
try:
    mitopath = pd.read_excel(MITOCARTA_FILE, sheet_name='C MitoPathways')
    pw_col, gn_col = 'MitoPathway', 'Genes'
except:
    mitopath = pd.read_excel(MITOCARTA_FILE, sheet_name=2)
    pw_col, gn_col = mitopath.columns[0], mitopath.columns[2]

gene_sets = {}
for _, row in mitopath.iterrows():
    if pd.notna(row[pw_col]) and pd.notna(row[gn_col]):
        genes = [g.strip() for g in str(row[gn_col]).split(',') if g.strip()]
        if len(genes) >= 5:
            gene_sets[str(row[pw_col])] = genes
print(f"MitoCarta pathways: {len(gene_sets)}")

print("Running GSEA (2-3 min)...")
gsea_results = gp.prerank(rnk=ranked_series, gene_sets=gene_sets,
                           min_size=5, max_size=500,
                           permutation_num=1000, seed=RANDOM_SEED, verbose=False)

gsea_df = gsea_results.res2d.copy()
gsea_df.columns = [c.strip() for c in gsea_df.columns]
clean_gsea = pd.DataFrame()
clean_gsea['Term'] = gsea_df.index if 'Term' not in gsea_df.columns else gsea_df['Term']
for c in gsea_df.columns:
    if c.upper()=='NES':    clean_gsea['NES'] = gsea_df[c].astype(float)
    if 'FDR' in c.upper(): clean_gsea['FDR'] = gsea_df[c].astype(float)
clean_gsea = clean_gsea.sort_values('FDR').reset_index(drop=True)
sig_gsea   = clean_gsea[clean_gsea['FDR'] < 0.25]

print(f"\n{len(sig_gsea)} pathways significant (FDR < 0.25):")
print(f"{'Pathway':<52} {'NES':>8}  {'FDR':>8}")
for _, r in sig_gsea.iterrows():
    print(f"{'↑' if r['NES']>0 else '↓'} {str(r['Term'])[:50]:<50} {r['NES']:>8.3f}  {r['FDR']:>8.4f}")

clean_gsea.to_csv(OUT + 'BRCA_GSEA_results.csv', index=False)
print("\nSaved GSEA results.")


## 15. Oncogenic Mutation Independence

In [ ]:
mutations  = pd.read_csv(MUTATION_FILE, sep='\t')
sample_col = 'sample'; gene_col = 'gene'
print(f"Mutation file: {mutations.shape}")

tp53_set   = set(mutations[mutations[gene_col]=='TP53'][sample_col].str[:15])
pik3ca_set = set(mutations[mutations[gene_col]=='PIK3CA'][sample_col].str[:15])
tp53_status   = pd.Series({c: int(c[:15] in tp53_set)   for c in tumor_cols})
pik3ca_status = pd.Series({c: int(c[:15] in pik3ca_set) for c in tumor_cols})
print(f"TP53 mutated:   {tp53_status.sum()} ({tp53_status.mean()*100:.1f}%)")
print(f"PIK3CA mutated: {pik3ca_status.sum()} ({pik3ca_status.mean()*100:.1f}%)")

mut_results = []
for gene, eid in CANDIDATES.items():
    expr_vals   = expression_tumor.loc[expression_tumor['Ensembl_clean']==eid, tumor_cols].values.flatten()
    expr_series = pd.Series(expr_vals, index=tumor_cols)
    for mut_name, status in [('TP53', tp53_status), ('PIK3CA', pik3ca_status)]:
        mutated  = expr_series[status==1].values
        wildtype = expr_series[status==0].values
        _, p = mannwhitneyu(mutated, wildtype, alternative='two-sided')
        mut_results.append({'gene':gene,'mutation':mut_name,
                            'n_mutated':len(mutated),'n_wildtype':len(wildtype),
                            'median_mutated':round(np.median(mutated),3),
                            'median_wildtype':round(np.median(wildtype),3),
                            'difference':round(np.median(mutated)-np.median(wildtype),3),
                            'pvalue':p})

mut_df = pd.DataFrame(mut_results)
_, adjm,_,_ = multipletests(mut_df['pvalue'], alpha=FDR_ALPHA, method='fdr_bh')
mut_df['adjusted_pvalue'] = adjm
mut_df['significant']     = mut_df['adjusted_pvalue'] < FDR_ALPHA
mut_df = mut_df.sort_values('adjusted_pvalue').reset_index(drop=True)

print(f"\nMutation significant: {mut_df['significant'].sum()}/20  (expected: 11)")
for _, r in mut_df.iterrows():
    sig = '***' if r['adjusted_pvalue']<0.001 else '**' if r['adjusted_pvalue']<0.01 else '*' if r['adjusted_pvalue']<0.05 else 'ns'
    print(f"  {r['gene']:<10} {r['mutation']:<8} {r['adjusted_pvalue']:<12.4e} {sig:<6} {r['difference']:+.3f}")

mut_df.to_csv(OUT + 'BRCA_mutation_correlation.csv', index=False)


## 16. Composite FAO Suppression Score

Weights = normalised |log₂FC| from TCGA all-stage (locked from Step 9).  
Score = weighted mean of z-scored expression. Built here; applied to GEO/METABRIC unchanged.


In [ ]:
cand_df  = de_all[de_all['ensembl_id'].isin(CANDIDATES.values())].copy()
cand_df['gene'] = cand_df['ensembl_id'].map({v:k for k,v in CANDIDATES.items()})
fc_vals  = cand_df.set_index('gene')['log2fc'].abs()
weights  = fc_vals / fc_vals.sum()
print("Composite score weights (normalised |log₂FC|):")
print(weights.sort_values(ascending=False).round(4))

def composite_score(expr_dict, ref_tumor_cols):
    # Compute weighted FAO suppression score for a set of samples.
    scores = {}
    for sample in expr_dict:
        sc = 0.0
        for gene, eid in CANDIDATES.items():
            t_e = expression_tumor.loc[expression_tumor['Ensembl_clean']==eid, ref_tumor_cols].values.flatten()
            s_e = expr_dict[sample].get(gene, np.nan)
            if np.isnan(s_e) or len(t_e)==0: continue
            z   = (s_e - np.mean(t_e)) / (np.std(t_e) + 1e-8)
            sc += weights.get(gene, 0) * z
        scores[sample] = sc
    return scores

# Build per-sample expression lookup for TCGA tumours and normals
def get_sample_exprs(sample_list, expr_df, cols):
    out = {}
    for samp in sample_list:
        d = {}
        for gene, eid in CANDIDATES.items():
            vals = expr_df.loc[expr_df['Ensembl_clean']==eid, [samp]].values.flatten()
            d[gene] = vals[0] if len(vals)>0 else np.nan
        out[samp] = d
    return out

t_expr_dict = get_sample_exprs(tumor_cols,  expression_tumor,  tumor_cols)
n_expr_dict = get_sample_exprs(normal_cols, expression_normal, tumor_cols)

score_dict   = composite_score(t_expr_dict, tumor_cols)
score_normal = composite_score(n_expr_dict, tumor_cols)

tumour_scores = np.array(list(score_dict.values()))
normal_scores = np.array(list(score_normal.values()))
_, p_score = mannwhitneyu(tumour_scores, normal_scores, alternative='two-sided')

print(f"\nComposite score — Tumour vs Normal:")
print(f"  Tumour mean = {tumour_scores.mean():.3f} (SD {tumour_scores.std():.3f})")
print(f"  Normal mean = {normal_scores.mean():.3f} (SD {normal_scores.std():.3f})")
print(f"  Mann-Whitney p = {p_score:.4e}")

# Composite score composite AUC
y_c = np.array([1]*len(tumour_scores)+[0]*len(normal_scores))
fpr_c, tpr_c, _ = roc_curve(y_c, np.concatenate([tumour_scores, normal_scores]))
comp_auc = auc(fpr_c, tpr_c)
if comp_auc < 0.5: comp_auc = 1 - comp_auc
print(f"  Composite AUC = {comp_auc:.4f}")

# Save
pd.DataFrame({'sample': list(score_dict.keys()), 'composite_score': list(score_dict.values()),
              'type': 'tumour'}).to_csv(OUT + 'BRCA_composite_scores.csv', index=False)
weights.to_csv(OUT + 'BRCA_composite_weights.csv')
print("Saved.")


In [ ]:
surv_score = surv_df.copy()
surv_score['composite_score'] = surv_score['barcode'].map(score_dict)
surv_score = surv_score.dropna(subset=['composite_score'])
med_score  = surv_score['composite_score'].median()
high_grp   = surv_score[surv_score['composite_score'] >  med_score]
low_grp    = surv_score[surv_score['composite_score'] <= med_score]

kmf_h = KaplanMeierFitter(); kmf_l = KaplanMeierFitter()
kmf_h.fit(high_grp['OS_time'], high_grp['OS_event'], label='High score (more suppression)')
kmf_l.fit(low_grp['OS_time'],  low_grp['OS_event'],  label='Low score')
res_score = logrank_test(high_grp['OS_time'], low_grp['OS_time'],
                          event_observed_A=high_grp['OS_event'], event_observed_B=low_grp['OS_event'])

fig, ax = plt.subplots(figsize=(8, 6))
kmf_h.plot_survival_function(ax=ax, ci_show=True, color='#c0392b', linewidth=2)
kmf_l.plot_survival_function(ax=ax, ci_show=True, color='#2196F3', linewidth=2)
ax.set_title(f'Composite FAO Score — Overall Survival (TCGA-BRCA)\nLog-rank p = {res_score.p_value:.3f}',
             fontweight='bold')
ax.set_xlabel('Time (days)'); ax.set_ylabel('Survival probability')
plt.tight_layout()
plt.savefig(OUT + 'BRCA_composite_score_KM.png', dpi=300)
plt.show()
print(f"Composite score KM log-rank p = {res_score.p_value:.4f}")


## 17. DESeq2 Methodological Cross-Validation

PyDESeq2 uses a negative binomial model on raw counts — independent of the Welch t-test.
Target: ≥ 80% sign concordance, Spearman ρ > 0.80. Result: 98.4%, ρ = 0.980.


In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds  import DeseqStats

print("Loading STAR raw counts...")
counts_raw = pd.read_csv(COUNTS_FILE, sep='\t', index_col=0)
counts_raw = counts_raw.drop(index=[i for i in counts_raw.index if str(i).startswith('N_')], errors='ignore')
counts_raw.index = counts_raw.index.str.split('.').str[0]
counts_mito = counts_raw[counts_raw.index.isin(mitocarta_set)].copy()

t_c = [c for c in counts_mito.columns if c[13:15]=='01']
n_c = [c for c in counts_mito.columns if c[13:15]=='11']
counts_sub = counts_mito[t_c+n_c].round(0).astype(int)
counts_sub = counts_sub[counts_sub.sum(axis=1)>0]
counts_T   = counts_sub.T
metadata   = pd.DataFrame({'condition':['tumor']*len(t_c)+['normal']*len(n_c)}, index=t_c+n_c)
metadata   = metadata.loc[counts_T.index]
print(f"DESeq2 matrix: {counts_T.shape} | Running...")

dds = DeseqDataSet(counts=counts_T, metadata=metadata,
                   design_factors='condition', ref_level=['condition','normal'], n_cpus=4, quiet=True)
dds.deseq2()
stat_res = DeseqStats(dds, contrast=['condition','tumor','normal'], alpha=0.05)
stat_res.summary()
deseq2_res = stat_res.results_df.copy()

# Concordance
de_all_dedup = de_all.drop_duplicates(subset='ensembl_id').set_index('ensembl_id')
ttest_sig_ids = set(de_sig['ensembl_id'])
deseq2_sig    = set(deseq2_res[deseq2_res['padj'] < PADJ_THRESHOLD].index)
agreement     = len(ttest_sig_ids & deseq2_sig) / max(len(ttest_sig_ids), 1)
common = de_all_dedup.index.intersection(deseq2_res.index)
tfc = de_all_dedup.loc[common,'log2fc'].values.astype(float)
dfc = deseq2_res.loc[common,'log2FoldChange'].values.astype(float)
mask = ~(np.isnan(tfc)|np.isnan(dfc))
rho, p_rho = spearmanr(tfc[mask], dfc[mask])

print(f"\nDESeq2 VALIDATION:")
print(f"  t-test significant:  {len(ttest_sig_ids)}")
print(f"  Confirmed by DESeq2: {len(ttest_sig_ids & deseq2_sig)} ({agreement*100:.1f}%)  (expected: 98.4%)")
print(f"  Spearman ρ:          {rho:.3f}  (expected: 0.980)")
print(f"  Genes in correlation: {mask.sum()}")
print(f"\nCandidate gene confirmation:")
for gene, eid in CANDIDATES.items():
    if eid in deseq2_res.index:
        r = deseq2_res.loc[eid]
        print(f"  {gene:<10} padj={r['padj']:.2e}  log2FC={r['log2FoldChange']:.2f}")

deseq2_res.to_csv(OUT + 'BRCA_DESeq2_all_results.csv')
print("\nSaved DESeq2 results.")


## 18. External Validation — GSE109169

25 matched tumour-normal pairs · Affymetrix Human Exon 1.0 ST (GPL5175) · Cross-platform.  
Full DE pipeline run independently — TCGA results not referenced.


In [ ]:
import GEOparse

geo_file = GEO109_FILE
print(f"Parsing: {os.path.basename(geo_file)}")

sample_sources = {}; sample_chars = {}; expr_lines = []; in_table = False

with gzip.open(geo_file, 'rt', encoding='utf-8', errors='replace') as f:
    for line in f:
        line = line.rstrip('\n')
        if line.startswith('!Sample_source_name_ch1'):
            for i,v in enumerate(re.findall(r'"([^"]+)"', line)): sample_sources[i] = v
        if line.startswith('!Sample_characteristics_ch1') and 'tissue:' in line:
            for i,v in enumerate(re.findall(r'"([^"]+)"', line)):
                if 'tissue:' in v: sample_chars[i] = v.split('tissue:')[-1].strip()
        if line.startswith('"ID_REF"'):
            col_names = line.strip().replace('"','').split('\t'); in_table = True; continue
        if in_table:
            if '!series_matrix_table_end' in line: break
            expr_lines.append(line)

geo_table = pd.read_csv(StringIO('\n'.join(expr_lines)), sep='\t', header=None, names=col_names)
geo_table = geo_table.set_index(col_names[0])
geo_sample_cols = col_names[1:]
for c in geo_sample_cols: geo_table[c] = pd.to_numeric(geo_table[c], errors='coerce')
print(f"Expression matrix: {geo_table.shape}  |  range: {geo_table.values.min():.2f}–{geo_table.values.max():.2f}")

tumor_geo  = [geo_sample_cols[i] for i,_ in enumerate(geo_sample_cols)
              if 'cancer' in sample_sources.get(i,'').lower() or 'tumor' in sample_sources.get(i,'').lower()]
normal_geo = [geo_sample_cols[i] for i,_ in enumerate(geo_sample_cols)
              if 'normal' in sample_sources.get(i,'').lower() or 'adjacent' in sample_sources.get(i,'').lower()]
print(f"Tumour: {len(tumor_geo)}  |  Normal: {len(normal_geo)}")

# ── GPL5175 probe → gene symbol mapping ────────────────────────────────────────
print("\nLoading GPL5175 annotation (may take a minute)...")
gpl = GEOparse.get_GEO("GPL5175", destdir=OUT, silent=True)
annot = gpl.table.copy()

def parse_symbol_huex(s):
    if not isinstance(s, str) or s.strip() == '---': return None
    parts = s.split(' // ')
    if len(parts) >= 2:
        sym = parts[1].strip()
        return sym if sym and sym != '---' else None
    return None

annot['Symbol'] = annot['gene_assignment'].apply(parse_symbol_huex)
annot_clean = annot[['ID','Symbol']].dropna(subset=['Symbol'])
annot_clean['ID'] = annot_clean['ID'].astype(str)
print(f"Probes with symbol: {len(annot_clean)}")

# Map probes, filter to MitoCarta, aggregate to median per gene
geo_table.index = geo_table.index.astype(str)
geo_table['Symbol'] = geo_table.index.map(annot_clean.set_index('ID')['Symbol'].to_dict())
mitocarta_symbols   = set(mitocarta_names['Symbol'].dropna())
geo_mito_agg = (geo_table[geo_table['Symbol'].isin(mitocarta_symbols)]
                .groupby('Symbol')[geo_sample_cols].median())
print(f"MitoCarta genes after aggregation: {len(geo_mito_agg)}")


In [ ]:
# PCA QC — GSE109169
geo_all_samples = [c for c in tumor_geo+normal_geo if c in geo_mito_agg.columns]
geo_matrix = geo_mito_agg[geo_all_samples].T.values.astype(float)
for col in range(geo_matrix.shape[1]):
    m = np.isnan(geo_matrix[:,col])
    if m.any(): geo_matrix[m,col] = np.nanmedian(geo_matrix[:,col])
var_mask   = geo_matrix.var(axis=0) > 0
geo_matrix = geo_matrix[:, var_mask]
geo_labels = ['Tumour']*len(tumor_geo) + ['Normal']*len(normal_geo)

geo_scaled = StandardScaler().fit_transform(geo_matrix)
geo_pca    = PCA(n_components=2, random_state=RANDOM_SEED)
geo_coords = geo_pca.fit_transform(geo_scaled)
geo_var1, geo_var2 = geo_pca.explained_variance_ratio_ * 100

fig, ax = plt.subplots(figsize=(7, 5))
for label, colour in [('Tumour','#c0392b'),('Normal','#1a7a4a')]:
    mask = [l==label for l in geo_labels]
    ax.scatter(geo_coords[mask,0], geo_coords[mask,1], c=colour, alpha=0.8, s=80,
               label=label, edgecolors='white', linewidths=0.5)
for i,(x,y) in enumerate(geo_coords):
    ax.annotate(str(i), (x,y), fontsize=6, alpha=0.6, xytext=(2,2), textcoords='offset points')
ax.set_xlabel(f'PC1 ({geo_var1:.1f}%)'); ax.set_ylabel(f'PC2 ({geo_var2:.1f}%)')
ax.set_title(f'PCA QC — GSE109169\n{len(tumor_geo)} tumour + {len(normal_geo)} normal', fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(OUT + 'GEO_PCA.png', dpi=300)
plt.show()

# Differential expression
geo_avail_t = [c for c in tumor_geo  if c in geo_mito_agg.columns]
geo_avail_n = [c for c in normal_geo if c in geo_mito_agg.columns]
geo109_results = []
for symbol in geo_mito_agg.index:
    t_v = geo_mito_agg.loc[symbol, geo_avail_t].values.astype(float)
    n_v = geo_mito_agg.loc[symbol, geo_avail_n].values.astype(float)
    t_v = t_v[~np.isnan(t_v)]; n_v = n_v[~np.isnan(n_v)]
    if len(t_v)<3 or len(n_v)<3: continue
    _, p = ttest_ind(t_v, n_v, equal_var=False)
    if np.isnan(p): p=1.0
    geo109_results.append({'Symbol':symbol,'log2fc':np.mean(t_v)-np.mean(n_v),'pvalue':p})

geo_de_109 = pd.DataFrame(geo109_results)
_, adjg,_,_ = multipletests(geo_de_109['pvalue'], alpha=0.05, method='fdr_bh')
geo_de_109['adjusted_pvalue'] = adjg
geo_de_109.to_csv(OUT + 'GEO_DE_all_genes.csv', index=False)

print(f"GSE109169 strict-sig: {((geo_de_109['adjusted_pvalue']<0.05)&(geo_de_109['log2fc'].abs()>1)).sum()}")

# Candidate replication
print(f"\n{'Gene':<12} {'TCGA FC':<12} {'GEO FC':<14} {'GEO adj p':<16} {'Replicated?'}")
print("-"*60)
for gene in CANDIDATES:
    t_row = de_all[de_all['Symbol']==gene]; g_row = geo_de_109[geo_de_109['Symbol']==gene]
    t_fc  = t_row['log2fc'].values[0] if len(t_row) else np.nan
    g_fc  = g_row['log2fc'].values[0] if len(g_row) else np.nan
    g_adj = g_row['adjusted_pvalue'].values[0] if len(g_row) else np.nan
    rep   = '✓' if (pd.notna(g_adj) and g_adj<0.05 and pd.notna(t_fc) and pd.notna(g_fc)
                    and np.sign(t_fc)==np.sign(g_fc)) else '✗'
    print(f"{gene:<12} {t_fc:<12.3f} {g_fc:<14.3f} {g_adj:<16.4e} {rep}")
# Expected: 10/10 replicated


In [ ]:
# Composite score — GSE109169
from scipy.stats import hypergeom

weights_loaded = pd.read_csv(OUT + 'BRCA_composite_weights.csv', index_col=0).squeeze()

geo109_t_scores = {}
for sample in geo_avail_t:
    sc = 0.0
    for gene in CANDIDATES:
        if gene in geo_mito_agg.index:
            gene_all = geo_mito_agg.loc[gene, geo_all_samples].values.astype(float)
            sv = geo_mito_agg.loc[gene, sample]
            z  = (sv - np.nanmean(gene_all)) / (np.nanstd(gene_all) + 1e-8)
            sc += weights_loaded.get(gene, 0) * z
    geo109_t_scores[sample] = sc

geo109_n_scores = {}
for sample in geo_avail_n:
    sc = 0.0
    for gene in CANDIDATES:
        if gene in geo_mito_agg.index:
            gene_all = geo_mito_agg.loc[gene, geo_all_samples].values.astype(float)
            sv = geo_mito_agg.loc[gene, sample]
            z  = (sv - np.nanmean(gene_all)) / (np.nanstd(gene_all) + 1e-8)
            sc += weights_loaded.get(gene, 0) * z
    geo109_n_scores[sample] = sc

t109 = np.array(list(geo109_t_scores.values()))
n109 = np.array(list(geo109_n_scores.values()))
_, p109 = mannwhitneyu(t109, n109, alternative='two-sided')
y109 = np.array([1]*len(t109)+[0]*len(n109))
fpr109, tpr109, _ = roc_curve(y109, np.concatenate([t109, n109]))
auc109 = auc(fpr109, tpr109)
if auc109 < 0.5: auc109 = 1 - auc109
print(f"GSE109169 composite AUC = {auc109:.3f}  |  p = {p109:.4e}")  # expected: AUC 0.998

# Save scores (with type column for figure generation)
pd.DataFrame({
    'sample': list(geo109_t_scores.keys())+list(geo109_n_scores.keys()),
    'score':  list(geo109_t_scores.values())+list(geo109_n_scores.values()),
    'type':   ['tumour']*len(geo109_t_scores) + ['normal']*len(geo109_n_scores)
}).to_csv(OUT + 'GEO_composite_scores.csv', index=False)

# Hypergeometric enrichment
core_symbols = set(core_df['Symbol'].dropna())
geo109_sig_syms = set(geo_de_109.loc[(geo_de_109['adjusted_pvalue']<0.05)&
                                      (geo_de_109['log2fc'].abs()>1),'Symbol'])
core_testable = [s for s in core_symbols if s in geo_mito_agg.index]
overlap109 = [s for s in core_testable if s in geo109_sig_syms]
M, n_sig, N, k = len(geo_mito_agg), len(geo109_sig_syms), len(core_testable), len(overlap109)
p_hyper = hypergeom.sf(k-1, M, n_sig, N)
concordant109 = sum(1 for s in overlap109
    if not de_all.loc[de_all['Symbol']==s,'log2fc'].empty
    and not geo_de_109.loc[geo_de_109['Symbol']==s,'log2fc'].empty
    and np.sign(de_all.loc[de_all['Symbol']==s,'log2fc'].values[0]) ==
       np.sign(geo_de_109.loc[geo_de_109['Symbol']==s,'log2fc'].values[0]))
print(f"Core testable: {N}  |  Replicated: {k}/{N}  |  p={p_hyper:.2e}")
print(f"Direction concordant: {concordant109}/{k} ({100*concordant109/max(k,1):.0f}%)")
# Expected: 19/43, hypergeometric p<10^-20, 19/19 concordant


## 19. External Validation — GSE42568

104 breast cancer tumours + 17 normal biopsies · Affymetrix HG-U133 Plus 2.0 (GPL570).  
Larger same-platform replication cohort.


In [ ]:
print("Parsing GSE42568 series matrix...")
sample_sources_42 = {}; expr_lines_42 = []; col_names_42 = []; in_table_42 = False

with gzip.open(GEO42_FILE, 'rt', encoding='utf-8', errors='replace') as f:
    for line in f:
        line = line.rstrip('\n')
        if line.startswith('!Sample_source_name_ch1'):
            for i,v in enumerate(re.findall(r'"([^"]+)"', line)): sample_sources_42[i] = v
        if line.startswith('"ID_REF"'):
            col_names_42 = line.strip().replace('"','').split('\t'); in_table_42=True; continue
        if in_table_42:
            if '!series_matrix_table_end' in line: break
            expr_lines_42.append(line)

geo42_expr = (pd.read_csv(StringIO('\n'.join(expr_lines_42)), sep='\t',
                          header=None, names=col_names_42)
              .set_index(col_names_42[0])
              .apply(pd.to_numeric, errors='coerce'))
geo42_sample_cols = col_names_42[1:]
print(f"Expression: {geo42_expr.shape}  range: {geo42_expr.values.min():.2f}–{geo42_expr.values.max():.2f}")

tumour_geo42 = [geo42_sample_cols[i] for i in range(len(geo42_sample_cols))
                if not any(k in sample_sources_42.get(i,'').lower()
                           for k in ['normal','non-cancer','healthy'])]
normal_geo42 = [geo42_sample_cols[i] for i in range(len(geo42_sample_cols))
                if any(k in sample_sources_42.get(i,'').lower()
                       for k in ['normal','non-cancer','healthy'])]
print(f"Tumour: {len(tumour_geo42)}  |  Normal: {len(normal_geo42)}")

print("\nLoading GPL570 annotation...")
gpl570   = GEOparse.get_GEO("GPL570", destdir=OUT, silent=True)
annot570 = gpl570.table.copy()

def first_symbol(s):
    if not isinstance(s, str) or s.strip() in ('', '---'): return None
    return s.split(' /// ')[0].strip()

annot570['Symbol'] = annot570['Gene Symbol'].apply(first_symbol)
probe_sym_map = annot570[['ID','Symbol']].dropna(subset=['Symbol']).set_index('ID')['Symbol'].to_dict()

geo42_expr.index = geo42_expr.index.astype(str)
geo42_expr['Symbol'] = geo42_expr.index.map(probe_sym_map)
geo42_mito = geo42_expr[geo42_expr['Symbol'].isin(mitocarta_symbols)].copy()
geo42_agg  = geo42_mito.groupby('Symbol')[geo42_sample_cols].median()
print(f"MitoCarta genes: {len(geo42_agg)}")

geo42_avail_t = [c for c in tumour_geo42 if c in geo42_agg.columns]
geo42_avail_n = [c for c in normal_geo42 if c in geo42_agg.columns]
geo42_all     = geo42_avail_t + geo42_avail_n


In [ ]:
# PCA QC — GSE42568
geo42_matrix = geo42_agg[geo42_all].T.values.astype(float)
for col in range(geo42_matrix.shape[1]):
    m = np.isnan(geo42_matrix[:,col])
    if m.any(): geo42_matrix[m,col] = np.nanmedian(geo42_matrix[:,col])
geo42_matrix = geo42_matrix[:, geo42_matrix.var(axis=0) > 0]
geo42_labels = ['Tumour']*len(geo42_avail_t) + ['Normal']*len(geo42_avail_n)

coords42 = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(
             StandardScaler().fit_transform(geo42_matrix))
v42 = PCA(n_components=2, random_state=RANDOM_SEED).fit(
          StandardScaler().fit_transform(geo42_matrix)).explained_variance_ratio_*100

fig, ax = plt.subplots(figsize=(7,5))
for label, colour in [('Tumour','#c0392b'),('Normal','#1a7a4a')]:
    mask = [l==label for l in geo42_labels]
    ax.scatter(coords42[mask,0], coords42[mask,1], c=colour, alpha=0.8, s=60,
               label=label, edgecolors='white', linewidths=0.5)
ax.set_xlabel(f'PC1 ({v42[0]:.1f}%)'); ax.set_ylabel(f'PC2 ({v42[1]:.1f}%)')
ax.set_title(f'PCA QC — GSE42568\n{len(geo42_avail_t)} tumour + {len(geo42_avail_n)} normal', fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout(); plt.savefig(OUT + 'GEO42_PCA.png', dpi=300); plt.show()

# DE
geo42_results = []
for symbol in geo42_agg.index:
    t_v = geo42_agg.loc[symbol, geo42_avail_t].values.astype(float)
    n_v = geo42_agg.loc[symbol, geo42_avail_n].values.astype(float)
    t_v = t_v[~np.isnan(t_v)]; n_v = n_v[~np.isnan(n_v)]
    if len(t_v)<3 or len(n_v)<3: continue
    _, p = ttest_ind(t_v, n_v, equal_var=False)
    if np.isnan(p): p=1.0
    geo42_results.append({'Symbol':symbol,'log2fc':np.mean(t_v)-np.mean(n_v),'pvalue':p})

geo42_de = pd.DataFrame(geo42_results)
_, adjg42,_,_ = multipletests(geo42_de['pvalue'], alpha=0.05, method='fdr_bh')
geo42_de['adjusted_pvalue'] = adjg42
geo42_de_sig = geo42_de[(geo42_de['adjusted_pvalue']<0.05)&(geo42_de['log2fc'].abs()>1)]
print(f"GSE42568 strict-sig: {len(geo42_de_sig)}")
geo42_de.to_csv(OUT + 'GEO42_DE_all_genes.csv', index=False)
geo42_de_sig.to_csv(OUT + 'GEO42_DE_significant.csv', index=False)

# Core replication
core_in_geo42  = [s for s in core_symbols if s in geo42_agg.index]
overlap_42     = [s for s in core_in_geo42 if s in set(geo42_de_sig['Symbol'])]
concordant_42  = sum(1 for s in overlap_42
    if not de_all.loc[de_all['Symbol']==s,'log2fc'].empty
    and not geo42_de.loc[geo42_de['Symbol']==s,'log2fc'].empty
    and np.sign(de_all.loc[de_all['Symbol']==s,'log2fc'].values[0]) ==
       np.sign(geo42_de.loc[geo42_de['Symbol']==s,'log2fc'].values[0]))
M42 = len(geo42_agg); n42 = len(geo42_de_sig); N42 = len(core_in_geo42); k42 = len(overlap_42)
p_hyper42 = hypergeom.sf(k42-1, M42, n42, N42)
print(f"Core testable: {N42}  |  Replicated: {k42}/{N42}  |  p={p_hyper42:.2e}")
print(f"Direction concordant: {concordant_42}/{k42}  (expected: 34/52, 100%)")

# Composite score — GSE42568
geo42_t_scores = {}
for sample in geo42_avail_t:
    sc = 0.0
    for gene in CANDIDATES:
        if gene in geo42_agg.index:
            gene_all = geo42_agg.loc[gene, geo42_all].values.astype(float)
            sv = geo42_agg.loc[gene, sample]
            z  = (sv - np.nanmean(gene_all)) / (np.nanstd(gene_all) + 1e-8)
            sc += weights_loaded.get(gene, 0) * z
    geo42_t_scores[sample] = sc

geo42_n_scores = {}
for sample in geo42_avail_n:
    sc = 0.0
    for gene in CANDIDATES:
        if gene in geo42_agg.index:
            gene_all = geo42_agg.loc[gene, geo42_all].values.astype(float)
            sv = geo42_agg.loc[gene, sample]
            z  = (sv - np.nanmean(gene_all)) / (np.nanstd(gene_all) + 1e-8)
            sc += weights_loaded.get(gene, 0) * z
    geo42_n_scores[sample] = sc

t42 = np.array(list(geo42_t_scores.values()))
n42s = np.array(list(geo42_n_scores.values()))
_, p42s = mannwhitneyu(t42, n42s, alternative='two-sided')
y42 = np.array([1]*len(t42)+[0]*len(n42s))
fpr42, tpr42, _ = roc_curve(y42, np.concatenate([t42, n42s]))
auc42 = auc(fpr42, tpr42)
if auc42 < 0.5: auc42 = 1 - auc42
print(f"GSE42568 composite AUC = {auc42:.3f}  |  p = {p42s:.4e}")  # expected: AUC 0.989

pd.DataFrame({
    'sample': list(geo42_t_scores.keys())+list(geo42_n_scores.keys()),
    'score':  list(geo42_t_scores.values())+list(geo42_n_scores.values()),
    'type':   ['tumour']*len(geo42_t_scores) + ['normal']*len(geo42_n_scores)
}).to_csv(OUT + 'GEO42_composite_scores.csv', index=False)


## 20. METABRIC Prognostic Validation

Applies TCGA-derived composite score unchanged to METABRIC (n=1,980) via cBioPortal API.  
Tests prognostic value independent of any DE replication.


In [ ]:
BASE  = "https://www.cbioportal.org/api"
STUDY = "brca_metabric"
PROF  = "brca_metabric_mrna_median_all_sample_Zscores"
HDRS  = {"Content-Type":"application/json","Accept":"application/json"}

print("Fetching METABRIC expression (cBioPortal API)...")
r = requests.post(f"{BASE}/genes/fetch", params={"geneIdType":"HUGO_GENE_SYMBOL"},
    json=list(CANDIDATES.keys()), headers=HDRS, timeout=30)
gene_info  = pd.DataFrame(r.json())
eid_to_sym = dict(zip(gene_info["entrezGeneId"], gene_info["hugoGeneSymbol"]))
eids       = gene_info["entrezGeneId"].tolist()

r = requests.post(f"{BASE}/molecular-profiles/{PROF}/molecular-data/fetch",
    json={"entrezGeneIds":eids,"sampleListId":"brca_metabric_all"},
    headers=HDRS, timeout=60)
expr_raw = pd.DataFrame(r.json())
val_col  = [c for c in expr_raw.columns if c.lower() in ["value","zscore","expression"]][0]
expr_raw["gene"] = expr_raw["entrezGeneId"].map(eid_to_sym)
mb_expr  = expr_raw.pivot(index="sampleId", columns="gene", values=val_col).reset_index()
mb_expr.columns.name = None
print(f"METABRIC expression: {mb_expr.shape}")

r = requests.get(f"{BASE}/studies/{STUDY}/clinical-data",
    params={"clinicalDataType":"PATIENT","projection":"DETAILED"}, timeout=60)
clin_pat = pd.DataFrame(r.json()).pivot(index="patientId", columns="clinicalAttributeId", values="value").reset_index()
r = requests.get(f"{BASE}/studies/{STUDY}/clinical-data",
    params={"clinicalDataType":"SAMPLE","projection":"DETAILED"}, timeout=60)
clin_sam = pd.DataFrame(r.json()).pivot(index="sampleId", columns="clinicalAttributeId", values="value").reset_index()
clin_sam["patientId"] = clin_sam["sampleId"].str.split("-").str[:2].str.join("-")
clin = clin_sam.merge(clin_pat, on="patientId", how="left")

metabric = mb_expr.merge(clin, on="sampleId", how="inner")
metabric["OS_event"] = metabric["OS_STATUS"].str.extract(r"^(\d)").astype(float)
metabric["OS_time"]  = pd.to_numeric(metabric["OS_MONTHS"], errors="coerce")
metabric["age"]      = pd.to_numeric(metabric["AGE_AT_DIAGNOSIS"], errors="coerce")
metabric["PAM50"]    = metabric["CLAUDIN_SUBTYPE"].map({"LumA":"LumA","LumB":"LumB","Basal":"Basal","Her2":"Her2"})
metabric["ER_bin"]   = (metabric["ER_STATUS"]=="Positive").astype(float)
print(f"METABRIC merged: {metabric.shape} | Events: {int(metabric['OS_event'].sum())}/{metabric['OS_event'].notna().sum()}")

# Apply composite score (METABRIC z-scored by cBioPortal — apply weights directly)
gene_cols_mb = [g for g in CANDIDATES if g in metabric.columns]
print(f"Candidate genes available: {len(gene_cols_mb)}/10")
metabric["composite_score"] = sum(
    weights_loaded.get(gene, 0) * pd.to_numeric(metabric[gene], errors='coerce').fillna(0)
    for gene in gene_cols_mb)


In [ ]:
# KM + Cox — METABRIC
surv_mb = metabric.dropna(subset=["OS_time","OS_event","composite_score"]).copy()
med_mb  = surv_mb["composite_score"].median()
high_mb = surv_mb[surv_mb["composite_score"] >  med_mb]
low_mb  = surv_mb[surv_mb["composite_score"] <= med_mb]

res_mb = logrank_test(high_mb["OS_time"], low_mb["OS_time"],
                       event_observed_A=high_mb["OS_event"], event_observed_B=low_mb["OS_event"])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax = axes[0]
kmf_h = KaplanMeierFitter(); kmf_l = KaplanMeierFitter()
kmf_h.fit(high_mb["OS_time"], high_mb["OS_event"], label=f"High (n={len(high_mb)})")
kmf_l.fit(low_mb["OS_time"],  low_mb["OS_event"],  label=f"Low (n={len(low_mb)})")
kmf_h.plot_survival_function(ax=ax, ci_show=True, color="#2196F3", linewidth=2)
kmf_l.plot_survival_function(ax=ax, ci_show=True, color="#e53935", linewidth=2)
ax.set_title(f"METABRIC — Composite FAO Score\nKM p = {res_mb.p_value:.3e}", fontweight="bold")
ax.set_xlabel("Time (months)"); ax.set_ylabel("Survival probability"); ax.set_ylim(0,1)

cox_mb  = surv_mb.dropna(subset=["age","PAM50","ER_bin"]).copy()
cox_mb["score_z"] = (cox_mb["composite_score"] - cox_mb["composite_score"].mean()) / cox_mb["composite_score"].std()
cox_mb  = pd.get_dummies(cox_mb, columns=["PAM50"], drop_first=True)
feat    = ["score_z","age","ER_bin"] + [c for c in cox_mb.columns if c.startswith("PAM50_")]
inp     = cox_mb[feat + ["OS_time","OS_event"]].dropna()
cph     = CoxPHFitter()
cph.fit(inp, duration_col="OS_time", event_col="OS_event")
s       = cph.summary.loc["score_z"]
HR      = round(s["exp(coef)"], 3)
p_cox   = s["p"]
ci_lo   = round(s["exp(coef) lower 95%"], 3)
ci_hi   = round(s["exp(coef) upper 95%"], 3)

ax2 = axes[1]
ax2.errorbar([HR], [0], xerr=[[HR-ci_lo],[ci_hi-HR]], fmt='o', color='#1a2e4a',
             markersize=10, capsize=6, linewidth=2, capthick=2)
ax2.axvline(1.0, color='k', linestyle='--', lw=1.2, alpha=0.6)
ax2.set_yticks([0]); ax2.set_yticklabels(["Composite\nFAO Score"], fontsize=11)
ax2.set_xlabel("Hazard Ratio (95% CI)")
ax2.set_title(f"METABRIC Cox (n={len(inp)}, {int(inp['OS_event'].sum())} events)\nAdjusted: age, PAM50, ER", fontweight="bold")
ax2.text(HR, 0.15, f"HR={HR}\n95% CI [{ci_lo}–{ci_hi}]\np={p_cox:.3e}",
         ha='center', va='bottom', fontsize=10, color='#1a2e4a')
ax2.spines["left"].set_visible(False)
plt.tight_layout()
plt.savefig(OUT + "METABRIC_composite_score_survival.png", dpi=300)
plt.show()

print(f"METABRIC KM p:   {res_mb.p_value:.4e}  (expected: ~7.8×10⁻⁷)")
print(f"METABRIC Cox HR: {HR}  (95% CI {ci_lo}–{ci_hi})  p={p_cox:.4e}")


## 21. GSEA — GSE42568 Independent Pathway Replication

In [ ]:
min_p_42 = geo42_de['adjusted_pvalue'][geo42_de['adjusted_pvalue']>0].min()
geo42_de['rank_score'] = (-np.log10(geo42_de['adjusted_pvalue'].replace(0, min_p_42))
                          * np.sign(geo42_de['log2fc']))
ranked_42 = (geo42_de.dropna(subset=['Symbol'])
             .sort_values('rank_score', ascending=False)
             .set_index('Symbol')['rank_score'])
ranked_42 = ranked_42[~ranked_42.index.duplicated(keep='first')]
print(f"GSE42568 ranked list: {len(ranked_42)} genes")

print("Running GSEA on GSE42568 (2-3 min)...")
gsea_42 = gp.prerank(rnk=ranked_42, gene_sets=gene_sets,
                      min_size=5, max_size=500,
                      permutation_num=1000, seed=RANDOM_SEED, verbose=False)

res_42 = gsea_42.res2d.copy()
res_42.columns = [c.strip() for c in res_42.columns]
clean_42 = pd.DataFrame()
clean_42['Term'] = res_42.index if 'Term' not in res_42.columns else res_42['Term']
for c in res_42.columns:
    if c.upper()=='NES':    clean_42['NES'] = res_42[c].astype(float)
    if 'FDR' in c.upper(): clean_42['FDR'] = res_42[c].astype(float)
clean_42 = clean_42.sort_values('FDR').reset_index(drop=True)
sig_42   = clean_42[clean_42['FDR'] < 0.25]

print(f"\nGSE42568: {len(sig_42)} pathways significant (FDR < 0.25)")
tcga_sig_terms = set(clean_gsea[clean_gsea['FDR']<0.25]['Term'].astype(str))
common_terms   = set(sig_42['Term'].astype(str)) & tcga_sig_terms
print(f"Replicated in both TCGA and GSE42568: {len(common_terms)}")

fao_row = clean_42[clean_42['Term'].str.contains('fatty acid', case=False, na=False)]
if len(fao_row):
    print(f"FAO pathway: NES={fao_row['NES'].values[0]:.3f}, FDR={fao_row['FDR'].values[0]:.4f}")
    # Expected: NES ≈ -2.064, FDR = 0.001

clean_42.to_csv(OUT + 'GEO42_GSEA_results.csv', index=False)


## 22. Manuscript Verification Summary

Checks all key numbers reported in the manuscript. Run this as the final step.


In [ ]:
print("=" * 65)
print("MANUSCRIPT VERIFICATION SUMMARY — BRCA")
print("=" * 65)

checks = []
def check(label, condition, note=""):
    sym = "✓" if condition else "✗ FAIL"
    print(f"  {sym}  {label}" + (f"  [{note}]" if note else ""))
    checks.append(condition)

print("\n── Cohort sizes ──")
check("Tumours = 1106",       len(tumor_cols) == 1106)
check("Normals = 113",        len(normal_cols) == 113)
check("Stage I indep = 163",  len(stage1_indep) == 163)
check("Stage I total = 184",  len(stage1_all) == 184)

print("\n── DE results ──")
check("All-stage significant = 126",  len(de_sig) == 126)
check("MitoCarta match = 1079",       len(de_all) >= 1079)

print("\n── Subtype stratification ──")
check("Pan-subtype intersection = 57", len(pan_ids) == 57)
check("Mixed-direction genes = 0",
      sum(1 for v in direction_map.values() if v=='MIXED') == 0)

print("\n── Core signature ──")
check("Core = 55 genes",  len(core_ids) == 55)

print("\n── Validation ──")
check("DESeq2 agreement ≥ 98%",  agreement >= 0.98)
check("DESeq2 Spearman ρ ≥ 0.97", rho >= 0.97)
check("GSE109169 AUC ≈ 0.998",   abs(auc109 - 0.998) < 0.003)
check("GSE42568 AUC ≈ 0.989",    abs(auc42  - 0.989) < 0.003)
check("All 10 candidates replicated in GSE109169",
      sum(1 for gene in CANDIDATES
          for g_row in [geo_de_109[geo_de_109['Symbol']==gene]]
          for t_row in [de_all[de_all['Symbol']==gene]]
          if len(g_row) and len(t_row)
          and g_row['adjusted_pvalue'].values[0]<0.05
          and np.sign(t_row['log2fc'].values[0])==np.sign(g_row['log2fc'].values[0]))
      == 10)
check("All 10 candidates replicated in GSE42568",
      sum(1 for gene in CANDIDATES
          for g_row in [geo42_de[geo42_de['Symbol']==gene]]
          for t_row in [de_all[de_all['Symbol']==gene]]
          if len(g_row) and len(t_row)
          and g_row['adjusted_pvalue'].values[0]<0.05
          and np.sign(t_row['log2fc'].values[0])==np.sign(g_row['log2fc'].values[0]))
      == 10)
check("METABRIC KM p < 0.001", res_mb.p_value < 0.001)

print(f"\n{'=' * 65}")
passed = sum(checks)
total  = len(checks)
print(f"RESULT: {passed}/{total} checks passed" +
      ("  ✓  ALL CORRECT" if passed==total else f"  ✗  {total-passed} FAILED"))
print("=" * 65)
